In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/gopikadevabalaji/world-happiness2023/world_happiness_2023.csv


In [2]:
import pandas as pd
import numpy as np

# World Happiness Report 2023 — representative data
# Source: https://www.kaggle.com/datasets/ajaypalsinghlo/world-happiness-report-2023

df = pd.read_csv('/kaggle/input/datasets/gopikadevabalaji/world-happiness2023/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']


print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())

Dataset: 63 countries, 9 columns
       Country                        Region  Happiness_Score     GDP  \
0      Finland                Western Europe            7.804  10.775   
1      Denmark                Western Europe            7.586  10.933   
2      Iceland                Western Europe            7.525  10.878   
3       Israel  Middle East and North Africa            7.473  10.527   
4  Netherlands                Western Europe            7.464  11.015   

   Social_Support  Life_Expectancy  Freedom  Generosity  Corruption  
0           0.954             71.9    0.949       0.142       0.179  
1           0.954             72.7    0.931       0.168       0.234  
2           0.983             72.5    0.961       0.260       0.150  
3           0.916             72.4    0.903       0.149       0.826  
4           0.939             72.4    0.879       0.240       0.296  


In [3]:
import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])

Regions in dataset:
Region
Western Europe                  15
Latin America and Caribbean     13
Central and Eastern Europe       7
Sub-Saharan Africa               7
Middle East and North Africa     6
North America and ANZ            4
Southeast Asia                   4
South Asia                       4
East Asia                        3
Name: count, dtype: int64

Score range: 1.859 – 7.804

Bottom 10 countries:
        Country                        Region  Happiness_Score
60  Afghanistan                    South Asia            1.859
61      Lebanon  Middle East and North Africa            2.392
62     Zimbabwe            Sub-Saharan Africa            2.995
52     Ethiopia            Sub-Saharan Africa            3.564
53     Tanzania            Sub-Saharan Africa            3.698
48   Bangladesh                    South Asia            3.892
47        India                    South Asia            4.036
50        Kenya            Sub-Saharan Africa            4.112
54       Uganda

In [4]:
import pandas as pd
import plotly.express as px

# Calculate average happiness score by region
region_avg = (
    df.groupby('Region')['Happiness_Score']
    .mean()
    .reset_index()
    .sort_values('Happiness_Score')
)

print(region_avg)

                         Region  Happiness_Score
5                    South Asia         3.618250
7            Sub-Saharan Africa         4.064714
3  Middle East and North Africa         4.943333
6                Southeast Asia         5.695250
2   Latin America and Caribbean         5.699000
1                     East Asia         5.966000
0    Central and Eastern Europe         6.338143
4         North America and ANZ         7.018250
8                Western Europe         7.085533


In [5]:
fig = px.bar(
    region_avg,
    x='Happiness_Score',
    y='Region',
    orientation='h',
    text='Happiness_Score',
    color='Happiness_Score',
    color_continuous_scale='Blues'
)

fig.update_traces(
    texttemplate='%{text:.2f}',
    textposition='outside'
)

fig.update_layout(
    title='Happiness Levels Vary Significantly Across World Regions',
    xaxis_title='Average Happiness Score',
    yaxis_title='',
    template='plotly_white',
    font=dict(size=14),
    xaxis=dict(
        range=[0, 10],
        showgrid=True,
        zeroline=True
    ),
    coloraxis_showscale=False,
    height=550,
    margin=dict(l=180, r=60, t=80, b=60)
)

fig.show()

In [6]:
import pandas as pd
import plotly.express as px

# Get top 8 happiest countries
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'

# Get bottom 8 least happy countries
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

# Combine both groups
combined = pd.concat([bottom8, top8])
combined = combined.sort_values('Happiness_Score')

# Calculate global average
global_avg = df['Happiness_Score'].mean()

print(f"Global average: {global_avg:.2f}")
print(combined[['Country', 'Happiness_Score', 'Group']])

Global average: 5.81
        Country  Happiness_Score     Group
60  Afghanistan            1.859  Bottom 8
61      Lebanon            2.392  Bottom 8
62     Zimbabwe            2.995  Bottom 8
52     Ethiopia            3.564  Bottom 8
53     Tanzania            3.698  Bottom 8
48   Bangladesh            3.892  Bottom 8
47        India            4.036  Bottom 8
50        Kenya            4.112  Bottom 8
7   Switzerland            7.240     Top 8
6        Norway            7.315     Top 8
5        Sweden            7.395     Top 8
4   Netherlands            7.464     Top 8
3        Israel            7.473     Top 8
2       Iceland            7.525     Top 8
1       Denmark            7.586     Top 8
0       Finland            7.804     Top 8


In [7]:
# Task 2: Top 8 vs Bottom 8 - Improved Chart

import plotly.express as px

gap = top8['Happiness_Score'].mean() - bottom8['Happiness_Score'].mean()

fig = px.bar(
    combined,
    x='Happiness_Score',
    y='Country',
    orientation='h',
    color='Group',
    text='Happiness_Score',
    color_discrete_map={
        'Top 8': '#159895',
        'Bottom 8': '#E76F51'
    },
    category_orders={
        'Country': combined['Country'].tolist()
    }
)

# Format labels
fig.update_traces(
    texttemplate='%{text:.2f}',
    textposition='outside',
    cliponaxis=False,
    marker_line_width=0
)

# Global average reference line
fig.add_vline(
    x=global_avg,
    line_dash='dash',
    line_color='#34495E',
    line_width=2
)

# Global average annotation inside chart
fig.add_annotation(
    x=global_avg,
    y=1.01,
    xref='x',
    yref='paper',
    text=f'Global average: {global_avg:.2f}',
    showarrow=False,
    xanchor='left',
    yanchor='bottom',
    font=dict(size=11, color='#34495E')
)

# Separator between groups
fig.add_hline(
    y=7.5,
    line_dash='dot',
    line_color='#999999',
    line_width=1.5
)

# Layout
fig.update_layout(
    title=dict(
        text=(
            f'<b>Top 8 Countries Score {gap:.2f} Points '
            'Higher Than Bottom 8</b>'
        ),
        x=0.02,
        xanchor='left',
        font=dict(size=18, color='#243B53')
    ),
    template='plotly_white',
    font=dict(size=12, color='#243B53'),
    xaxis=dict(
        title='Happiness Score',
        range=[0, 9],
        dtick=1,
        showgrid=True,
        gridcolor='#E8EDF2',
        zeroline=True
    ),
    yaxis=dict(
        title='',
        automargin=True
    ),
    legend=dict(
        title='',
        orientation='h',
        x=0,
        y=1.10,
        xanchor='left',
        yanchor='bottom'
    ),
    bargap=0.25,
    width=1000,
    height=750,
    margin=dict(
        l=140,
        r=80,
        t=150,
        b=60
    ),
    paper_bgcolor='white',
    plot_bgcolor='white'
)

fig.show()